# Gemini Enterprise ↔ Snowflake MCP via Entra ID (External OAuth)

This quickstart connects **Gemini Enterprise (GE)** to a **Snowflake-managed MCP server** using **Microsoft Entra ID** as the external OAuth identity provider.

## Architecture

```
GE  →  Entra ID (authz + token)  →  Snowflake External OAuth  →  MCP Server  →  Cortex Analyst
```

## Prerequisites

| Requirement | Details |
|-------------|----------|
| Snowflake account | With `ACCOUNTADMIN` or `SECURITYADMIN` + `USERADMIN` access |
| Semantic view | Already created and working with Cortex Analyst |
| MCP server | Already created (see base `mcp-connection-quickstart`) |
| Azure tenant | Admin access to Microsoft Entra ID (Azure AD) |
| GE access | Gemini Enterprise with Custom MCP Connector enabled |

# Part 1: Azure (Entra ID) Setup

Complete these steps in the **Azure Portal** before running the Snowflake cells.

## 1.1 Register the App (or use an existing one)

1. **Azure Portal** → **App registrations** → **New registration**
2. Name: e.g. `Snowflake-MCP-Agent`
3. Supported account types: **Single tenant**
4. Redirect URI: **Web** → `https://vertexaisearch.cloud.google.com/oauth-redirect`
5. Click **Register**

> If you already have the app registered, just ensure the redirect URI is added (see 1.2).

## 1.2 Add the GE Redirect URI

This is the URI Gemini Enterprise uses to complete the OAuth callback.

1. Go to your app → **Authentication**
2. Under **Web** → **Redirect URIs**, add:
   ```
   https://vertexaisearch.cloud.google.com/oauth-redirect
   ```
3. Click **Save**

> Without this, sign-in will fail with `AADSTS500113: No reply address is registered`.

## 1.3 Create a Client Secret

1. Go to your app → **Certificates & secrets** → **Client secrets** → **New client secret**
2. Description: e.g. `GE MCP connector`
3. Expiry: choose per your policy
4. Copy the **Value** immediately (it won't be shown again)

## 1.4 Expose an API (Snowflake OAuth Resource)

This app (or a separate resource app) must expose a scope that Snowflake will validate.

1. Go to the **resource** app → **Expose an API**
2. Set **Application ID URI** if not already set (e.g. `api://<resource-app-client-id>`)
3. **Add a scope**:
   - Scope name: `session:role-any`
   - Who can consent: **Admins and users**
   - Display name: `Snowflake session with any role`
4. Click **Add scope**

## 1.5 Grant API Permissions + Admin Consent

1. Go back to the **client** app → **API permissions**
2. **Add a permission** → **My APIs** → select the resource app → **Delegated** → check `session:role-any`
3. Click **Grant admin consent for [tenant]** → **Yes**

## 1.6 Create a Test User

1. **Azure Portal** → **Microsoft Entra ID** → **Users** → **New user** → **Create new user**
2. Set a UPN (e.g. `testuser@yourdomain.onmicrosoft.com`) and password
3. This UPN must exactly match the Snowflake user's `LOGIN_NAME` (see Part 3)

> You may also use an existing user in the tenant.

# Part 2: Collect Your Values

After completing Part 1, gather these values. Replace the placeholders in the SQL cells below.

| Placeholder | Where to find it | Example |
|-------------|-----------------|----------|
| `<TENANT_ID>` | Azure Portal → App registrations → Overview → Directory (tenant) ID | `b3b06c45-b6f1-...` |
| `<CLIENT_ID>` | Azure Portal → App registrations → Overview → Application (client) ID | `15f8b4f7-016b-...` |
| `<CLIENT_SECRET>` | Azure Portal → Certificates & secrets → Value column | `NUL8Q~fDx...` |
| `<RESOURCE_APP_ID_URI>` | Azure Portal → Resource app → Expose an API → Application ID URI | `api://9eac3d7f-...` |
| `<SNOWFLAKE_ACCOUNT>` | Snowflake account locator + region | `myaccount.us-central1.gcp` |
| `<ENTRA_USER_UPN>` | The test user's User Principal Name from Part 1.6 | `user@contoso.onmicrosoft.com` |
| `<DATABASE.SCHEMA>` | Database and schema where the MCP server lives | `POC.AI` |
| `<MCP_SERVER_NAME>` | Name of the existing MCP server | `GE_WEATHER_MCP_SERVER` |
| `<SEMANTIC_VIEW>` | Fully-qualified semantic view name | `POC.AI.WEATHER_MAN` |

In [ ]:
%%sql -r semantic_view_check
DESCRIBE SEMANTIC VIEW <DATABASE.SCHEMA>.<SEMANTIC_VIEW_NAME>;

# Part 3: Snowflake — Create External OAuth Integration

This tells Snowflake to trust tokens issued by your Entra ID tenant.

In [ ]:
%%sql -r external_oauth_create_result
CREATE OR REPLACE SECURITY INTEGRATION GE_MCP_ENTRA_EXTERNAL_OAUTH
  TYPE = EXTERNAL_OAUTH
  ENABLED = TRUE
  EXTERNAL_OAUTH_TYPE = AZURE
  EXTERNAL_OAUTH_ISSUER = 'https://sts.windows.net/<TENANT_ID>/'
  EXTERNAL_OAUTH_JWS_KEYS_URL = 'https://login.microsoftonline.com/<TENANT_ID>/discovery/v2.0/keys'
  EXTERNAL_OAUTH_AUDIENCE_LIST = ('<RESOURCE_APP_ID_URI>')
  EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = 'upn'
  EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'login_name'
  EXTERNAL_OAUTH_ANY_ROLE_MODE = 'ENABLE';

In [ ]:
%%sql -r external_oauth_description
DESCRIBE SECURITY INTEGRATION GE_MCP_ENTRA_EXTERNAL_OAUTH;

# Part 4: Snowflake — Map the Entra User

The External OAuth integration maps the token's `upn` claim to a Snowflake user's `login_name`. The user must exist and have an appropriate role.

> Use `USERADMIN` or `ACCOUNTADMIN` role to run the next cell.

In [ ]:
%%sql -r user_mapping_result
USE ROLE USERADMIN;

CREATE USER IF NOT EXISTS "<ENTRA_USER_UPN>"
  LOGIN_NAME = '<ENTRA_USER_UPN>'
  DEFAULT_ROLE = <ROLE_FOR_MCP>;

GRANT ROLE <ROLE_FOR_MCP> TO USER "<ENTRA_USER_UPN>";

# Part 5: Verify MCP Server

In [ ]:
%%sql -r mcp_servers_list
SHOW MCP SERVERS IN SCHEMA <DATABASE.SCHEMA>;

In [ ]:
%%sql -r mcp_server_description
DESCRIBE MCP SERVER <DATABASE.SCHEMA>.<MCP_SERVER_NAME>;

# Part 6: Register in Gemini Enterprise

Create a Custom MCP connection in GE with these values:

## Connection Settings

| Field | Value |
|-------|-------|
| **Connection name** | `mcp-snowflake-cortex-entra` |
| **MCP endpoint** | `https://<SNOWFLAKE_ACCOUNT>.snowflakecomputing.com/api/v2/databases/<db>/schemas/<schema>/mcp-servers/<MCP_SERVER_NAME>` |
| **Authorization URL** | `https://login.microsoftonline.com/<TENANT_ID>/oauth2/v2.0/authorize` |
| **Token URL** | `https://login.microsoftonline.com/<TENANT_ID>/oauth2/v2.0/token` |
| **Client ID** | `<CLIENT_ID>` |
| **Client Secret** | `<CLIENT_SECRET>` |
| **Scope** | `<RESOURCE_APP_ID_URI>/session:role-any` |

> If the primary endpoint doesn't work, try appending `/sse`.

## Description

```text
Snowflake-managed MCP server that lets Gemini Enterprise query Snowflake analytics through Cortex Analyst via a natural-language MCP tool.
```

## Agent Instructions

```text
Use this MCP server to answer questions about data stored in Snowflake.

When responding:
- Use the available analyst tool for data questions.
- Prefer direct factual answers first, then include brief supporting context when useful.
- If the question is ambiguous, ask a short clarifying question before using the tool.
- If the tool returns no result, say the dataset could not answer the question.
- Do not invent values, dates, or calculations not returned by the tool.
- Keep answers concise and business-friendly.
```

# Part 7: Verify Connection in GE

Click **Verify connection** in GE. The flow:

1. A Microsoft sign-in dialog opens
2. Sign in with the Entra user from Part 1.6 / Part 4
3. On first sign-in you may be prompted to change your password and set up MFA/text verification — complete these steps
4. GE should show the connection as verified

## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| `AADSTS500113: No reply address is registered` | GE redirect URI not added to app | Add `https://vertexaisearch.cloud.google.com/oauth-redirect` in app → Authentication (Part 1.2) |
| `AADSTS700016: Application not found` | Wrong tenant or client ID | Double-check `<TENANT_ID>` and `<CLIENT_ID>` |
| `AADSTS65001: User needs consent` | Admin consent not granted | Grant admin consent (Part 1.5) |
| Auth succeeds but Snowflake rejects token | User not mapped | Ensure Snowflake user `LOGIN_NAME` matches the token's `upn` claim exactly (Part 4) |
| Network timeout / generic auth error | Network policy blocking | Run the optional diagnostic cells below |

## Optional: Network Policy Diagnostic

> Only run if GE auth fails with a generic error after successful Microsoft sign-in. This proves whether a Snowflake network policy is blocking the token exchange.

> **Not for production** — replace with a scoped policy after diagnosis.

In [ ]:
%%sql -r temp_network_policy_result
CREATE OR REPLACE NETWORK POLICY TEMP_GE_ENTRA_OAUTH_ALLOW_ALL
  ALLOWED_IP_LIST = ('0.0.0.0/0');

In [ ]:
%%sql -r attach_policy_result
ALTER SECURITY INTEGRATION GE_MCP_ENTRA_EXTERNAL_OAUTH
  SET NETWORK_POLICY = TEMP_GE_ENTRA_OAUTH_ALLOW_ALL;

In [ ]:
%%sql -r integration_after_policy
DESCRIBE SECURITY INTEGRATION GE_MCP_ENTRA_EXTERNAL_OAUTH;

# Part 8: Test End-to-End

Once the connection is verified, open a GE chat session and try:

```text
What was the hottest day in NYC in 2021?
```

```text
How much snowfall was recorded in January 2021?
```

Success criteria:
- GE routes the question through the MCP connector (not a generic answer)
- The response is grounded in the Snowflake dataset
- If auth is green but GE doesn't use the tool, the issue is GE tool discovery, not Snowflake OAuth

# Cleanup

Remove the temporary network policy after diagnosis:

```sql
ALTER SECURITY INTEGRATION GE_MCP_ENTRA_EXTERNAL_OAUTH
  UNSET NETWORK_POLICY;

DROP NETWORK POLICY IF EXISTS TEMP_GE_ENTRA_OAUTH_ALLOW_ALL;
```

To remove the entire Entra integration:

```sql
DROP SECURITY INTEGRATION IF EXISTS GE_MCP_ENTRA_EXTERNAL_OAUTH;
DROP USER IF EXISTS "<ENTRA_USER_UPN>";
```